# Radlogistik — Pipeline Explained

This notebook explains and demonstrates every stage of the Radlogistik pipeline.
All outputs are saved to `notebook_data/`.

**Outputs:**
- `notebook_data/poi_map.html` — POI overview map with automatic connection layers
- `notebook_data/loop_map.html` — delivery loop map (producer + consumer layers; custom layers if a `loops.json` path is supplied)
- `notebook_data/loops/*.geojson` — GeoJSON exports of computed loops

## 0 — Project structure

```
claude_map/
├── main.py                      ← project-specific entry point (custom rules for Havixbeck)
├── pipeline.py                  ← generic reusable pipeline — the one used by this notebook
├── osm_download.py              ← OSM POI downloader via Overpass API
├── data/
│   ├── points_of_interest.csv   ← curated POI list (custom for Havixbeck)
│   ├── goods.csv                ← goods catalogue with potential scores
│   └── loops.json               ← optional custom delivery loops (project-specific)
├── osm_downloader/
│   └── osm_poi_config.json      ← config: OSM tag→POI type→goods mapping
├── aoi/aoi.gpkg                 ← area-of-interest polygon
├── streets/
│   ├── streets.osm              ← raw OSM XML (downloaded once)
│   └── streets.graphml          ← cached igraph-ready street graph
└── src/
    ├── connections.py           ← all-pairs connection builder + igraph routing
    ├── routing.py               ← speed, travel-time, bike-friendliness
    ├── delivery_loops.py        ← producer / consumer / custom loop builders
    ├── route_map.py             ← Folium map rendering
    ├── route_map_scripts.js     ← JavaScript injected into the interactive map
    ├── route_map_styles.css     ← CSS for the interactive map
    ├── osm.py                   ← OSM attribute parsers (bike lanes, pavement, …)
    └── co2.py                   ← CO₂ emission model (HBEFA)
```

> **Note**: `main.py` adds project-specific rules on top of the generic pipeline
> (special POI colours, custom delivery loops). This notebook uses `pipeline.py`
> directly so the output is clean and transferable to other areas.

## 1 — Automatic OSM data download

### 1.1 Area of interest (AOI)

The pipeline operates inside a geographic polygon. Two cells below let you pick it interactively:

1. Type any place name and geocode it via **Nominatim** (`osmnx.geocode_to_gdf`) — a
   list of matching places is shown so you can confirm the right one was found.
2. Optionally **redraw** the boundary by hand on an interactive map (rectangle,
   polygon, or circle tool) — useful when the geocoded boundary is too coarse or
   covers the wrong area. If nothing is drawn, the geocoded polygon is kept as-is.

The resulting polygon is saved to `notebook_data/aoi.gpkg` and used for every
subsequent step (OSM download, street graph, POI loading, pipeline run).

Requires `ipyleaflet` and `geopy` (`pip install ipyleaflet geopy`).

### 1.2 `osm_poi_config.json` — mapping OSM tags to goods

In [ ]:
import os
import osmnx as ox
import geopandas as gpd
from geopy.geocoders import Nominatim

os.makedirs('notebook_data', exist_ok=True)


def get_city_geometry(place_name: str) -> gpd.GeoDataFrame:
    """Geocode a place name to a polygon via Nominatim (through osmnx)."""
    return ox.geocode_to_gdf(place_name)


def get_geographic_suggestions_from_string(place_name: str, user_agent: str = 'radlogistik_notebook'):
    """Return every Nominatim match for `place_name` so you can confirm the right one was found."""
    geolocator = Nominatim(user_agent=user_agent)
    return geolocator.geocode(place_name, exactly_one=False)


# ── USER INPUT ──────────────────────────────────────────────────────────────
city_name = "Havixbeck, Germany"

aoi = get_city_geometry(city_name)
# aoi = gpd.read_file('path/to/your_aoi.gpkg')  # Alternative: load your own AOI file

geo_suggestions = get_geographic_suggestions_from_string(city_name)
geo_suggestions  # check that the right place was found

In [ ]:
import ipyleaflet
from ipyleaflet import Map, DrawControl, GeoData
from shapely.geometry import shape


def ipyleaflet_drawable_map(m=None, center=(0, 0), zoom=11, height='800px'):
    """Interactive map with drawing controls.

    Returns (m, get_drawn_geometries) where get_drawn_geometries() returns a
    GeoDataFrame of everything drawn, or None if nothing was drawn.
    """
    if m is None:
        m = Map(center=center, zoom=zoom, scroll_wheel_zoom=True, layout={'height': height})
        google_hybrid = ipyleaflet.TileLayer(
            url='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
            name='Google Hybrid', attribution='Google', opacity=0.75,
        )
        m.add_layer(google_hybrid)

    draw_control = DrawControl(
        rectangle={'shapeOptions': {'color': 'red'}},
        polygon={'shapeOptions': {'color': 'red'}},
        circle={'shapeOptions': {'color': 'red'}},
        polyline={}, marker={}, circlemarker={},
    )
    m.add_control(draw_control)

    drawn_geometries = {}

    def handle_draw(target, action, geo_json):
        layer_id = geo_json.get('id')
        if action in ('created', 'edited'):
            drawn_geometries[layer_id] = shape(geo_json['geometry'])
            print(f'Geometry {action}: {layer_id}')
        elif action == 'deleted':
            drawn_geometries.pop(layer_id, None)
            print(f'Geometry deleted: {layer_id}')

    draw_control.on_draw(handle_draw)

    def get_drawn_geometries():
        if not drawn_geometries:
            return None
        return gpd.GeoDataFrame(geometry=list(drawn_geometries.values()), crs='EPSG:4326')

    return m, get_drawn_geometries


center = aoi.to_crs(4326).union_all().centroid
m, get_aoi = ipyleaflet_drawable_map(center=[center.y, center.x])
geo_data = GeoData(
    geo_dataframe=aoi.to_crs(4326),
    style={'color': 'blue', 'fillColor': 'blue', 'opacity': 1, 'fillOpacity': 0, 'weight': 4, 'dashArray': '5,5'},
)
m.add_layer(geo_data)
m

In [ ]:
# Run this AFTER drawing (or not drawing) on the map above.
new_aoi = get_aoi()
if new_aoi is None:
    print('Nothing drawn — keeping the geocoded AOI.')
else:
    aoi = new_aoi
    print('Using the hand-drawn AOI.')

aoi_path = 'notebook_data/aoi.gpkg'
aoi.to_crs(4326).to_file(aoi_path, driver='GPKG')
print(f'AOI saved to {aoi_path}')
aoi

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import osm_download

os.makedirs('notebook_data', exist_ok=True)

download_config = {
    'topics':         ['food', 'gastronomy', 'healthcare', 'pharmacy'],
    'config_dir':     '../osm_downloader',
    'aoi_path':       aoi_path,               # from the interactive AOI selection above
    'output_pois':    'notebook_data/pois_osm.csv',
    'output_goods':   'notebook_data/goods_osm.csv',
}

osm_download.run(download_config)

In [ ]:
import requests

# Download the raw street network (OSM XML) for the AOI via Overpass, so the
# pipeline can build its own igraph-ready street graph from it (see Section 3).
_OVERPASS_URLS = [
    'https://overpass-api.de/api/interpreter',
    'https://lz4.overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
]

streets_osm_path = 'notebook_data/streets.osm'

if os.path.exists(streets_osm_path):
    print(f'{streets_osm_path} already exists — skipping download.')
else:
    bounds = aoi.to_crs(4326).total_bounds  # (minx, miny, maxx, maxy)
    bbox_str = f'{bounds[1]},{bounds[0]},{bounds[3]},{bounds[2]}'
    query = f'''
    [out:xml][timeout:180];
    (
      way["highway"]({bbox_str});
    );
    (._;>;);
    out body;
    '''
    for i, url in enumerate(_OVERPASS_URLS, 1):
        try:
            resp = requests.get(url, params={'data': query}, timeout=180,
                                 headers={'User-Agent': 'radlogistik_notebook'})
            resp.raise_for_status()
            with open(streets_osm_path, 'w', encoding='utf-8') as f:
                f.write(resp.text)
            print(f'Saved street network to {streets_osm_path} (server {i})')
            break
        except Exception as e:
            print(f'  server {i} failed: {e} — trying next')
    else:
        raise RuntimeError('All Overpass servers failed to deliver the street network.')

## 2 — Points of interest & goods catalogue

The pipeline accepts either the auto-downloaded CSV (above) or the
hand-curated `data/points_of_interest.csv`. Both use semicolons.

### Key columns

| column | type | meaning |
|---|---|---|
| `poi_id` | int | unique ID |
| `Lat` / `Lon` | float | WGS-84 |
| `Sector` | str | category (e.g. gastronomy, supermarket, farm, bakery) |
| `Company` | str | business name |
| `ConsumedGoods` | list[int] | good IDs this POI **receives** |
| `ProducedGoods` | list[int] | good IDs this POI **delivers** |

### Goods catalogue (`data/goods.csv`)

Good IDs are integers defined here. The pipeline converts them to names before
any processing, so the CSV stores numeric IDs but all internal logic uses strings.

| ID | name | icon | potential |
|---|---|---|---|
| 0 | drinks | 🥤 | 8 |
| 1 | food | 🧺 | 9 |
| 2 | bread | 🍞 | 8 |
| 3 | wine | 🍷 | 8 |
| 4 | beer | 🍺 | 7 |
| 5 | honey | 🍯 | 9 |
| 14 | manufactured_products | 🏭 | 6 |
| 15 | agricultural_products | 🌾 | 8 |
| 22 | asparagus | 🌿 | 8 |
| 23 | strawberries | 🍓 | 8 |

In [ ]:
import pandas as pd

# Use the curated POI file; swap path to 'notebook_data/pois_osm.csv' for the auto-downloaded one
pois_df = pd.read_csv('../data/points_of_interest.csv', sep=';')
print(f'{len(pois_df)} POIs')
pois_df[['poi_id', 'Company', 'Sector', 'ConsumedGoods', 'ProducedGoods']].head(15)

In [ ]:
goods_df = pd.read_csv('../data/goods.csv')
goods_df[['good_id', 'Product', 'Potential', 'Weight', 'Size', 'Icon']]

## 3 — Street graph & routing attributes

### 3.1 Graph loading

The OSM XML is parsed by **osmnx** into a directed multigraph, projected to the
local UTM zone, and reduced to its largest strongly-connected component so that
every node is reachable from every other node.

Custom edge attributes added after loading:

| attribute | values | source |
|---|---|---|
| `bike_separation` | none / soft / complete / mixed / prohibited | `src/osm.py` |
| `access_restrictions` | pedestrian+bikes / private / permit / residents | `src/osm.py` |
| `pavement` | asphalt / concrete / cobblestone / unpaved | `src/osm.py` |
| `highway` | normalised set | `src/routing.py` |

### 3.2 Car routing

Speed per edge comes from the OSM `maxspeed` tag with per-highway-type fallbacks:

```python
car_maxspeeds = {
    'living_street': 30,  'residential': 30,
    'secondary':     40,  'tertiary':    40,
    'primary':       50,  'motorway':   100,
    'trunk':         80,  'service':     20,
    'unclassified':  40,
}
```

Travel time uses a **trapezoidal speed profile** (acceleration + cruise + deceleration)
via `routing.travel_time()`. Key parameters from `main.py`:

| parameter | value | meaning |
|---|---|---|
| `car_acceleration` | 1.5 m/s² | urban acceleration |
| `car_min_cruising_time` | 5 s | minimum cruise phase |
| `car_node_penalty` | 5 s | delay per intersection |
| `car_stopping_time` | 1 min | loading/unloading at each stop |

CO₂ per edge is estimated via `co2.route_hbefa()` using HBEFA hot-emission
factors for a gasoline passenger car.

### 3.3 E-bike routing

E-bikes use lower, strictly-enforced speed caps (`enforce=True`):

```python
ebike_maxspeeds = {
    'living_street': 15, 'residential': 15,
    'secondary':     18, 'tertiary':    15,
    'primary':       20, 'motorway':     1,  # effectively blocked
    'unclassified':  15,
}
```

Motorways and trunk roads receive speed 1 km/h so the router naturally avoids them.

### 3.4 Perceived travel time & bike-friendliness

The e-bike routing objective is **perceived** travel time, not raw time.
Unfriendly road segments feel subjectively longer:

```
worst = 1 / (1 − max_travel_time_reduction)       # = 1.67 at 40 %
t_perceived = t_raw × (worst − (friendliness − 1) × (worst − 1) / 9)
```

Segments with `bikefriendliness == 0` are multiplied by `_BIKE_AVOID_FACTOR = 50`
to make them practically unreachable without being hard-blocked.

**Bike-friendliness score** (1–10) is a weighted combination of six OSM attributes:

| attribute | weight | best value (score = 10) |
|---|---|---|
| `bike_separation` | 10 | complete (dedicated lane) |
| `highway` type | 10 | cycleway, track, residential |
| `access_restrictions` | 10 | pedestrian+bikes |
| `lanes` | 10 | 1 lane |
| `car_maxspeed` | 10 | ≤ 20 km/h |
| `pavement` | 5 | asphalt / concrete |

In [ ]:
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import src.routing as routing
import src.osm as osm

# Project the AOI to a local UTM CRS (needed for graph projection + distance math)
aoi = aoi.to_crs(aoi.estimate_utm_crs())

streets_graph_path = 'notebook_data/streets.graphml'

if os.path.exists(streets_graph_path):
    G = ox.load_graphml(streets_graph_path)
else:
    # Build the igraph-ready street graph from the downloaded OSM XML, the
    # same way pipeline.run() does it — see Section 3.1.
    G = ox.graph_from_xml(streets_osm_path)
    G = ox.project_graph(G, to_crs=aoi.crs)
    G = ox.truncate.largest_component(G, strongly=True)
    nodes, edges = ox.graph_to_gdfs(G)

    network_gdf = ox.features_from_xml(filepath=streets_osm_path)
    edges['bike_separation']     = osm.bike_separation(edges, network_gdf)
    edges['access_restrictions'] = osm.access_restrictions(edges, network_gdf)
    edges['pavement']            = osm.pavement(edges, network_gdf)
    edges['all_highways']        = edges['highway'].copy()
    edges['highway']             = edges['highway'].map(routing.normalize_route_type)
    G = ox.graph_from_gdfs(nodes, edges)
    ox.save_graphml(G, streets_graph_path)

nodes, edges = ox.graph_to_gdfs(G)
print(f'Nodes: {len(nodes):,}   Edges: {len(edges):,}')

car_maxspeeds = {
    'living_street': 30, 'motorway': 100, 'motorway_link': 60,
    'primary': 50, 'primary_link': 50, 'residential': 30,
    'secondary': 40, 'secondary_link': 40, 'service': 20,
    'tertiary': 40, 'tertiary_link': 40, 'trunk': 80,
    'trunk_link': 60, 'unclassified': 40,
}
bf_cfg = {
    'bike_separation':    {'column':'bike_separation','weight':10,'default':1,'mode':'categorical','list_behaviour':'max',
                           'values':{'none':1,'mixed':1,'complete':10,'soft':5,'prohibited':1},'ignore':{'prohibited':['access_restrictions','highway','lanes','car_maxspeed']}},
    'highway':            {'column':'highway','weight':10,'default':1,'mode':'categorical','list_behaviour':'max',
                           'values':{'cycleway':10,'track':10,'residential':10,'living_street':10,'path':10,'service':8,
                                     'footway':7,'pedestrian':7,'tertiary':5,'tertiary_link':5,'secondary':3,
                                     'unclassified':1,'primary':1,'trunk':1,'motorway':1},
                           'ignore':{'living_street':['bike_separation','car_maxspeed'],'residential':['bike_separation','car_maxspeed'],
                                     'cycleway':['bike_separation','car_maxspeed','lanes']}},
    'lanes':              {'column':'lanes','weight':10,'default':5,'mode':'numeric','list_behaviour':'max','values':{1:10,2:5,3:1}},
    'car_maxspeed':       {'column':'car_maxspeed','weight':10,'default':5,'mode':'numeric','list_behaviour':'max',
                           'values':{0:10,5:10,20:8,30:5,50:1},'ignore':{0:['bike_separation']}},
    'pavement':           {'column':'pavement','weight':5,'default':10,'mode':'categorical','list_behaviour':'max',
                           'values':{'asphalt':10,'concrete':10,'cobblestone':5,'unpaved':1}},
    'access_restrictions':{'column':'access_restrictions','weight':10,'default':5,'mode':'categorical','list_behaviour':'max',
                           'values':{'pedestrian+bikes':10,'private':1,'pedestrian':1}},
}

edges['car_maxspeed']   = routing.infer_maxspeed(edges, car_maxspeeds, enforce=False)
edges['bikefriendliness'] = edges.apply(
    lambda r: routing.compute_bikefriendliness(r, bf_cfg, min_bikefriendliness=5), axis=1)

edges['bikefriendliness'].hist(bins=10, edgecolor='black')
plt.title('Bike-friendliness distribution')
plt.xlabel('Score (1–10)'); plt.ylabel('Edge count')
plt.tight_layout()
plt.savefig('notebook_data/bikefriendliness_dist.png', dpi=150)
plt.show()
print(edges['bikefriendliness'].describe().round(2))

## 4 — igraph routing & connection builder

### 4.1 igraph shortest paths

The osmnx graph is converted to an **igraph** directed graph for fast
shortest-path computation. For each source POI the router calls
`get_shortest_paths()` twice — once weighted by `car_travel_time`,
once by `ebike_percieved_travel_time` — in a single batch.

Edge geometries are assembled in traversal order with **direction correction**:
osmnx stores one geometry per edge regardless of direction, so the code checks
which endpoint is entered and reverses the coordinate sequence if needed.

### 4.2 Connection builder (`connections.py`)

`build_all_pairs_connections()` generates one row per (origin, destination, product) where:
- origin **produces** the product
- destination **consumes** the product
- goods `potential > 0`

**Same-sector filter**: if origin and destination belong to the same sector
group, a connection is only allowed if the destination does **not** itself
produce the product. This prevents redundant same-sector deliveries
(e.g. supermarket → supermarket for food both already stock).

The food sector group combines supply-chain partners:
```python
_FOOD_SECTOR_GROUP = frozenset({
    'Lebensmittelhandel', 'Landwirtschaft',
    'supermarket', 'bakery', 'food_market', 'drink_shop',
    'winery', 'brewery', 'beekeeper',
})
```

In [ ]:
import geopandas as gpd
import pandas as pd
from src.data_utils import safe_parse_list, is_list_column
from src.connections import build_all_pairs_connections
import src.routing as routing

goods = pd.read_csv('../data/goods.csv')
for c in [col for col in goods.columns if is_list_column(goods[col])]:
    goods[c] = goods[c].apply(safe_parse_list)

pois = pd.read_csv('../data/points_of_interest.csv', sep=';')
for col in ['ConsumedGoods', 'ProducedGoods']:
    pois[col] = pois[col].apply(safe_parse_list)
pois['Lat'] = pd.to_numeric(pois['Lat'], errors='coerce')
pois['Lon'] = pd.to_numeric(pois['Lon'], errors='coerce')
pois = gpd.GeoDataFrame(pois, geometry=gpd.points_from_xy(pois['Lon'], pois['Lat']), crs=4326)
pois = pois.to_crs(aoi.crs)

id_to_name = {int(r['good_id']): str(r['Product']) for _, r in goods.iterrows()}
for col in ['ConsumedGoods', 'ProducedGoods']:
    pois[col] = pois[col].apply(
        lambda ids: [id_to_name[int(x)] for x in (ids or []) if str(x).lstrip('-').isdigit() and int(x) in id_to_name]
    )

pois['osmid'] = routing.nearest_nodes(pois, G)
pois['poi_id'] = pois['poi_id'].astype(int)
pois = pois.set_index('poi_id', drop=False)

connections_df = build_all_pairs_connections(pois=pois, goods=goods)
print(f'{len(connections_df):,} connections  |  '
      f'{connections_df["origin"].nunique()} origins  |  '
      f'{connections_df["destination"].nunique()} destinations  |  '
      f'{connections_df["Product"].nunique()} products')
connections_df[['origin', 'destination', 'Product', 'Potential']].head(10)

## 5 — Delivery loops

### 5.1 Automatic producer loops

Starting from each supply POI the algorithm adds consumer stops within a radius,
subject to constraints:

| parameter | value | meaning |
|---|---|---|
| `MAX_RADIUS` | 5 000 m | maximum straight-line distance from producer |
| `MAX_STOPS` | 12 | maximum consumer stops per loop |
| `MAX_ADDED_DISTANCE` | 250 m | max detour to add one more stop |

Stop order is optimised by a nearest-neighbour heuristic.

### 5.2 Automatic consumer loops

Starting from each demand POI the algorithm visits all suppliers of each
consumed good, again respecting `MAX_STOPS` and `MAX_ADDED_DISTANCE`.

### 5.3 Custom loops (`loops.json`) — optional

If a `loops.json` path is supplied, these layers appear **in addition** to the
automatic producer and consumer layers. Custom loops are useful for representing
real supply chains that the automatic algorithm cannot infer from POI metadata alone.

**Format:**

```json
[
  {
    "layer_name": "My supply chain",
    "loops": [
      [
        {"poi_id": 32, "load_products": [15, 22, 23], "unload_products": [], "mandatory": true},
        {"poi_id": 43, "load_products": [], "unload_products": [15, 22, 23], "mandatory": false},
        {"poi_id": 44, "load_products": [], "unload_products": [15, 22, 23], "mandatory": false},
        {"poi_id": 32, "load_products": [], "unload_products": [], "mandatory": true}
      ]
    ]
  }
]
```

**Leg semantics:**

| field | meaning |
|---|---|
| `load_products` | good IDs **picked up** at this stop (this POI produces them) |
| `unload_products` | good IDs **dropped off** at this stop (this POI consumes them) |
| `mandatory: true` | the loop is hidden if this stop cannot be routed |

A loop always starts and ends at the **home POI** (the first leg's `poi_id`).
The `allRouteable` check in `route_map_scripts.js` validates that every consecutive
stop pair has a route in the PAIRS dictionary before displaying the loop.

## 6 — Full pipeline run

Set `CUSTOM_LOOPS_PATH` to `None` to get only the automatic producer/consumer
layers, or point it to a `loops.json` file to add custom layers.

> This notebook uses `pipeline.run()` directly — it does **not** apply any of
> the project-specific rules from `main.py` (no special POI colours, no
> hardcoded supplier assignments).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pipeline
import src.route_map as _rm

# Disable project-specific POI colour + icon rules so the notebook produces a
# clean, generic map (these only make sense for the Havixbeck main.py study —
# e.g. collapsing Gastronomie/supermarket consumed-goods lists to a single 🧺
# basket icon, or colouring a fixed set of restaurant/Tilbeck POIs).
_rm._RESTAURANT_POIS      = set()
_rm._TILBECK_POIS         = set()
_rm._BASKET_ICON_ENABLED  = False

os.makedirs('notebook_data', exist_ok=True)

# Set to None for automatic layers only, or point to a loops.json for custom layers
CUSTOM_LOOPS_PATH = None
# CUSTOM_LOOPS_PATH = '../data/loops.json'   # ← uncomment to include custom layers

config = {
    'paths': {
        'pois':          '../data/points_of_interest.csv',
        'goods':         '../data/goods.csv',
        'aoi':           'notebook_data/aoi.gpkg',
        'streets_graph': 'notebook_data/streets.graphml',
        'osm_xml':       'notebook_data/streets.osm',
        'streets':       'notebook_data',
        'custom_loops':  CUSTOM_LOOPS_PATH or '',
        'loops_output':  'notebook_data/loops',
        'map_output':    'notebook_data',
        'raster_output': 'raster_layers',
    },
    'raster_dpi': 150,
    'loop': {
        'max_stops':            12,
        'max_radius_m':         5000,
        'max_added_distance_m': 250,
    },
    'car': {
        'node_penalty':          5,
        'acceleration':          1.5,
        'min_cruising_time':     5,
        'min_cruising_speed':    10,
        'max_stop_and_go_speed': 50,
        'stopping_time':         1,
        'maxspeeds': {
            'living_street': 30, 'motorway': 100, 'motorway_link': 60,
            'primary': 50, 'primary_link': 50, 'residential': 30,
            'secondary': 40, 'secondary_link': 40, 'service': 20,
            'tertiary': 40, 'tertiary_link': 40, 'trunk': 80,
            'trunk_link': 60, 'unclassified': 40,
        },
    },
    'ebike': {
        'node_penalty':          1,
        'acceleration':          2,
        'min_cruising_time':     5,
        'min_cruising_speed':    5,
        'max_stop_and_go_speed': 0,
        'stopping_time':         0.1,
        'bike_avoid_factor':     50,
        'maxspeeds': {
            'living_street': 15, 'motorway': 1, 'motorway_link': 1,
            'primary': 20, 'primary_link': 20, 'residential': 15,
            'secondary': 18, 'secondary_link': 18, 'service': 15,
            'tertiary': 15, 'tertiary_link': 15, 'trunk': 20,
            'trunk_link': 20, 'unclassified': 15,
        },
    },
    'scoring': {
        'min_bikefriendliness':      5,
        'max_travel_time_reduction': 0.4,
        'max_bike_extra_time':       0.1,
        'friendliness_weight':       3,
        'time_weight':               4,
        'product_weight':            3,
    },
    'bikefriendliness': {
        'access_restrictions': {'column':'access_restrictions','weight':10,'default':5,'mode':'categorical',
                                'list_behaviour':'max',
                                'values':{'pedestrian+bikes':10,'private':1,'pedestrian':1,'permit':10,'residents':10},
                                'ignore':{'pedestrian+bikes':['bike_separation','highway','lanes','car_maxspeed'],
                                          'private':['bike_separation','highway','lanes','car_maxspeed'],
                                          'pedestrian':['bike_separation','highway','lanes','car_maxspeed']}},
        'bike_separation': {'column':'bike_separation','weight':10,'default':1,'mode':'categorical',
                            'list_behaviour':'max',
                            'values':{'none':1,'mixed':1,'complete':10,'soft':5,'prohibited':1},
                            'ignore':{'prohibited':['access_restrictions','highway','lanes','car_maxspeed']}},
        'pavement': {'column':'pavement','weight':5,'default':10,'mode':'categorical','list_behaviour':'max',
                     'values':{'asphalt':10,'concrete':10,'cobblestone':5,'unpaved':1}},
        'highway': {'column':'highway','weight':10,'default':1,'mode':'categorical','list_behaviour':'max',
                    'values':{'trunk':1,'motorway_link':1,'trunk_link':1,'secondary':3,'path':10,
                              'unclassified':1,'tertiary':5,'service':8,'track':10,'residential':10,
                              'living_street':10,'footway':7,'cycleway':10,'primary':1,'motorway':1,
                              'tertiary_link':5,'pedestrian':7,'secondary_link':3,'bridleway':8},
                    'ignore':{'living_street':['bike_separation','car_maxspeed'],
                              'residential':['bike_separation','car_maxspeed'],
                              'footway':['bike_separation','car_maxspeed','lanes'],
                              'cycleway':['bike_separation','car_maxspeed','lanes']}},
        'lanes': {'column':'lanes','weight':10,'default':5,'mode':'numeric','list_behaviour':'max',
                  'values':{1:10,2:5,3:1}},
        'car_maxspeed': {'column':'car_maxspeed','weight':10,'default':5,'mode':'numeric','list_behaviour':'max',
                         'values':{0:10,5:10,20:8,30:5,50:1},'ignore':{0:['bike_separation']}},
    },
    'default_language': 'de',
}

pipeline.run(config)
print('\nOutputs written to notebook_data/')

## 7 — Exploring outputs

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

loop_dir = 'notebook_data/loops'
loop_files = sorted(f for f in os.listdir(loop_dir) if f.endswith('.geojson'))
print('Exported loop files:', loop_files)

if loop_files:
    gdf = gpd.read_file(os.path.join(loop_dir, loop_files[0]))
    print(f'\n{loop_files[0]}: {len(gdf)} loops')
    display(gdf[['home_poi', 'products', 'car_time_min', 'ebike_time_min', 'car_dist_km']].head(10))

    fig, ax = plt.subplots(figsize=(10, 8))
    gdf[gdf.geometry.notna()].plot(ax=ax, color='steelblue', linewidth=1.5, alpha=0.7)
    ax.set_title(loop_files[0].replace('_loops.geojson', '').replace('_', ' ').title())
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig('notebook_data/loops_preview.png', dpi=150)
    plt.show()

In [ ]:
import pathlib

for fname in ['poi_map.html', 'loop_map.html']:
    p = pathlib.Path('notebook_data') / fname
    if p.exists():
        print(f'{fname}: {p.resolve()}')